```bash
CUDA_VISIBLE_DEVICES=7 vllm serve Qwen/Qwen2.5-7B-Instruct \
    --host 0.0.0.0 \
    --port 8084 \
    --gpu-memory-utilization 0.85 \
    --enable-prefix-caching \
    --dtype bfloat16 \
    --max_model_len 32000 \
    --trust-remote-code
```

```bash
CUDA_VISIBLE_DEVICES=6 vllm serve Skywork/Skywork-o1-Open-PRM-Qwen-2.5-7B \
    --host 0.0.0.0 \
    --port 8082 \
    --gpu-memory-utilization 0.8 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

```bash
CUDA_VISIBLE_DEVICES=5 vllm serve Qwen/Qwen2.5-Math-PRM-7B \
    --host 0.0.0.0 \
    --port 8083 \
    --gpu-memory-utilization 0.8 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

In [95]:
from openai import OpenAI

OPENAI_API_KEY = "EMPTY"
OPENAI_API_BASE = "http://localhost:{PORT}/v1"

causal_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE.format(PORT=8084),
)
causal_model = causal_client.models.list().data[0].id

skywork_prm_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE.format(PORT=8082),
)
skywork_prm_model = skywork_prm_client.models.list().data[0].id

qwen_prm_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE.format(PORT=8083),
)
qwen_prm_model = qwen_prm_client.models.list().data[0].id


In [96]:
import re
import pandas as pd
from tqdm.auto import tqdm
from functools import partial
from transformers import AutoTokenizer
from multiprocessing import Pool, cpu_count
from constants.prompts_constants import (
    VERBOSE_TASK, CONSISE_TASK, EQ_TO_TEXT_TASK, CHANGE_NUMBERS_TASK
)

from utils.prompt_utils import get_augmentation_prompt, get_equivalence_prompt
from utils.io_utils import prepare_input, derive_step_rewards_vllm, prepare_batch_input_for_model

In [97]:
df = pd.read_parquet("data/processbench.parquet")

df.keys()

Index(['id', 'generator', 'problem', 'steps', 'final_answer_correct', 'label',
       'split', 'steps_len', 'per_step_len', 'Qwen2.5-Math-PRM-7B',
       'Skywork-o1-Open-PRM-Qwen-2.5-7B'],
      dtype='object')

In [98]:
def augmentor(df, task_text, client, model):
    # _, row = index_row
    prompts = []
    tokenizer = AutoTokenizer.from_pretrained(model)

    for index_row in df.iterrows():
        _, row = index_row
        question, steps = row["problem"], row["steps"]
        prompt          = get_augmentation_prompt(question, steps, task_text)
        prompt = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, tokenize=False)
        prompts.append(prompt)

    # call OpenAI
    responses = []
    batch_size = 256
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch = prompts[i:i + batch_size]
        resp = client.completions.create(
            model=model,
            prompt=batch,
            max_tokens=4000,
            temperature=0.9,
        ).choices
        responses.extend(resp)
    resp = responses
    contents = sorted(resp, key=lambda x: int(x.index))
    contents = [content.text for content in contents]
    list_aug_questions = []
    list_aug_steps = []

    for content in contents:

        # grab the <response>…</response> block
        m = re.search(r"<response>(.*?)</response>", content, re.DOTALL)
        if not m:
            # return {"aug_question":"", "aug_steps":[]}
            list_aug_questions.append("")
            list_aug_steps.append([])
            continue

        body = m.group(1).strip()

        # extract question
        q_m = re.search(r"<question>(.*?)</question>", body, re.DOTALL)
        aug_question = q_m.group(1).strip() if q_m else ""
        list_aug_questions.append(aug_question)

        # extract steps: find all <step#>…</step#>
        aug_steps = re.findall(r"<step\d+>(.*?)</step\d+>", body, re.DOTALL)
        aug_steps = [s.strip() for s in aug_steps]
        list_aug_steps.append(aug_steps)

    return {"aug_question": list_aug_questions, "aug_steps": list_aug_steps}

def equivalence_check(df_org, df_aug, client, model):
    # (_, rowA), (_, rowB) = pair
    tokenizer = AutoTokenizer.from_pretrained(model)
    qAs, stepsAs = df_org["problem"], df_org["steps"]
    qBs, stepsBs = df_aug["aug_question"], df_aug["aug_steps"]

    prompts = []

    for qA, stepsA, qB, stepsB in zip(qAs, stepsAs, qBs, stepsBs):
        prompt = get_equivalence_prompt(qA, stepsA, qB, stepsB)
        prompt = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, tokenize=False)
        prompts.append(prompt)

    responses = []
    batch_size = 256
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch = prompts[i:i + batch_size]
        resp = client.completions.create(
            model=model,
            prompt=batch,
            max_tokens=4000,
        ).choices
        responses.extend(resp)
    resp = responses
    contents = sorted(resp, key=lambda x: int(x.index))
    contents = [content.text for content in contents]
    equivalence_results = []
    body_equivalence_results = []

    for content in contents:

        # extract the <response>…</response> block
        m = re.search(r"<response>(.*?)</response>", content, re.DOTALL)
        if not m:
            equivalence_results.append(False)
            body_equivalence_results.append(content)
            continue

        body = m.group(1)
        body_equivalence_results.append(body)
        # grab question flag
        q_m = re.search(r"<question>\s*([YN])\s*</question>", body)
        question_flag = q_m.group(1) if q_m else "N"

        # grab all step flags into a list
        step_flags = re.findall(r"<step\d+>\s*([YN])\s*</step\d+>", body)

        # final check: question + every step must be "Y"
        all_flags = [question_flag] + step_flags
        all_flags = all(f == "Y" for f in all_flags)
        equivalence_results.append(all_flags)
    return {"equivalence": equivalence_results, "body_equivalence_results": body_equivalence_results}

def prm_scorer(questions, steps, client, model, batch_size=32):
    num_samples = len(questions)
    tokenizer = AutoTokenizer.from_pretrained(model)

    input_ids_all = []
    token_mask_all = []
    all_rewards = []

    for (question, step) in tqdm(zip(questions, steps), total=len(questions), desc="[PRM] Preparing input"):
        input_ids, token_mask = prepare_input(
                                model, 
                                problem=question, 
                                steps=step, 
                                tokenizer=tokenizer,
                                convert_to_list=True
        )
        input_ids_all.append(input_ids)
        token_mask_all.append(token_mask)


    for start_idx in tqdm(range(0, num_samples, batch_size), desc="[PRM] Scoring"):
        end_idx = start_idx + batch_size
    
        batch_input_ids = input_ids_all[start_idx:end_idx]
        batch_token_masks = token_mask_all[start_idx:end_idx]
    
        batch_input_ids, batch_token_masks = prepare_batch_input_for_model(batch_input_ids, batch_token_masks, pad_token_id=0)
    
        batch_logits = client.embeddings.create(
            input=batch_input_ids.cpu().tolist(),
            model=model,
        )
    
        rewards = derive_step_rewards_vllm(
            model,
            batch_logits,
            batch_token_masks,
            tokenizer
        )
    
        all_rewards.extend(rewards)
    return all_rewards

In [99]:
# def augmentor(index_row, task_text, client, model):
#     _, row = index_row

#     question, steps = row["problem"], row["steps"]
#     prompt          = get_augmentation_prompt(question, steps, task_text)

#     # call OpenAI
#     resp = client.chat.completions.create(
#         model=model,
#         messages=[
#             {"role": "user", "content": prompt}
#         ],

#     )
#     content = resp.choices[0].message.content

#     # grab the <response>…</response> block
#     m = re.search(r"<response>(.*?)</response>", content, re.DOTALL)
#     if not m:
#         return {"aug_question":"", "aug_steps":[]}

#     body = m.group(1).strip()

#     # extract question
#     q_m = re.search(r"<question>(.*?)</question>", body, re.DOTALL)
#     aug_question = q_m.group(1).strip() if q_m else ""

#     # extract steps: find all <step#>…</step#>
#     aug_steps = re.findall(r"<step\d+>(.*?)</step\d+>", body, re.DOTALL)
#     aug_steps = [s.strip() for s in aug_steps]

#     return {"aug_question": aug_question, "aug_steps": aug_steps}

# def equivalence_check(pair, client, model):
#     (_, rowA), (_, rowB) = pair
#     qA, stepsA = rowA["problem"], rowA["steps"]
#     qB, stepsB = rowB["aug_question"], rowB["aug_steps"]

#     prompt = get_equivalence_prompt(qA, stepsA, qB, stepsB)
#     resp = client.chat.completions.create(
#         model=model,
#         messages=[{"role": "user", "content": prompt}],
#     )
#     content = resp.choices[0].message.content

#     # extract the <response>…</response> block
#     m = re.search(r"<response>(.*?)</response>", content, re.DOTALL)
#     if not m: return False

#     body = m.group(1)

#     # grab question flag
#     q_m = re.search(r"<question>\s*([YN])\s*</question>", body)
#     question_flag = q_m.group(1) if q_m else "N"

#     # grab all step flags into a list
#     step_flags = re.findall(r"<step\d+>\s*([YN])\s*</step\d+>", body)

#     # final check: question + every step must be "Y"
#     all_flags = [question_flag] + step_flags
#     return all(f == "Y" for f in all_flags)

# def prm_scorer(questions, steps, client, model, batch_size=32):
#     num_samples = len(questions)
#     tokenizer = AutoTokenizer.from_pretrained(model)

#     input_ids_all = []
#     token_mask_all = []
#     all_rewards = []

#     for (question, step) in tqdm(zip(questions, steps), total=len(questions), desc="[PRM] Preparing input"):
#         input_ids, token_mask = prepare_input(
#                                 model, 
#                                 problem=question, 
#                                 steps=step, 
#                                 tokenizer=tokenizer,
#                                 convert_to_list=True
#         )
#         input_ids_all.append(input_ids)
#         token_mask_all.append(token_mask)


#     for start_idx in tqdm(range(0, num_samples, batch_size), desc="[PRM] Scoring"):
#         end_idx = start_idx + batch_size
    
#         batch_input_ids = input_ids_all[start_idx:end_idx]
#         batch_token_masks = token_mask_all[start_idx:end_idx]
    
#         batch_input_ids, batch_token_masks = prepare_batch_input_for_model(batch_input_ids, batch_token_masks, pad_token_id=0)
    
#         batch_logits = client.embeddings.create(
#             input=batch_input_ids.cpu().tolist(),
#             model=model,
#         )
    
#         rewards = derive_step_rewards_vllm(
#             model,
#             batch_logits,
#             batch_token_masks,
#             tokenizer
#         )
    
#         all_rewards.extend(rewards)
#     return all_rewards

In [100]:
def attack(df_sample, 
    task_text, 
    prm_client, 
    experiment_name, 
    causal_client=causal_client
):
    causal_model = causal_client.models.list().data[0].id
    prm_model = prm_client.models.list().data[0].id

    # Apply the augmentor function to each row in the DataFrame
    aug_results = []
    # for index_row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Augmenting"):
    #     aug_results.append(augmentor(index_row, task_text, client=causal_client, model=causal_model))
    aug_results = augmentor(df_sample, task_text, client=causal_client, model=causal_model)
    df_aug = pd.DataFrame(aug_results)

    # Check equivalence
    equivalence_results = []
    # for pair in tqdm(zip(df_sample.iterrows(), df_aug.iterrows()), total=len(df_sample), desc="Checking equivalence"):
    #     equivalence_results.append(equivalence_check(pair, client=causal_client, model=causal_model))
    equivalence_results = equivalence_check(df_sample, df_aug, client=causal_client, model=causal_model)
    df_aug["equivalence"] = equivalence_results["equivalence"]
    df_aug["body_equivalence_results"] = equivalence_results["body_equivalence_results"]

    # PRM Scorer
    rewards = prm_scorer(questions=df_sample["problem"].tolist(),
                        steps=df_aug["aug_steps"].tolist(), 
                        client=prm_client, model=prm_model)
    df_aug[f"{prm_model}--rewards"] = rewards
    
    # concat the original and augmented DataFrames
    for key in df_aug.keys():
        df_sample[key] = df_aug[key]
    
    # Save the DataFrame to a CSV file
    df_sample.to_parquet(f"experiments/{experiment_name}.parquet", index=False)
    return df_sample

In [101]:
len(df)

3400

In [ ]:
sample_size = 2
df_sample = df.sample(sample_size, random_state=42).reset_index(drop=True)

attack(df, 
    task_text=VERBOSE_TASK, 
    prm_client=skywork_prm_client, 
    experiment_name="skywork_verbose_task_2"
)

  0%|          | 0/14 [00:00<?, ?it/s]

In [78]:
experiment_name = "skywork_verbose_task_2"
df_sample = pd.read_parquet(f"experiments/{experiment_name}.parquet")
print(df_sample["body_equivalence_results"].iloc[0])

<response>
    <question> Y </question>
    <step1>


In [ ]:
experiment_name = "verbose_task_5"
df_sample = pd.read_parquet(f"experiments/{experiment_name}.parquet")
df_sample.head()

,id,generator,problem,steps,final_answer_correct,label,split,steps_len,per_step_len,Qwen2.5-Math-PRM-7B,Skywork-o1-Open-PRM-Qwen-2.5-7B,aug_question,aug_steps,equivalence,Qwen/Qwen2.5-Math-PRM-7B--rewards
0,omnimath-299,Llama-3.1-70B-Instruct,Calculate the sum: $\sum_{n=1}^{99} \left(n^{3...,"[To calculate the given sum, we can start by e...",False,3,omnimath,6,"[444, 297, 332, 293, 279, 176]","[0.671875, 0.9921875, 0.98828125, 0.0903320312...","[0.5193994365554163, 0.34864513533394575, 0.39...","Given the series to calculate, find the sum of...","[To solve this, we start by observing the expr...",True,"[0.96875, 0.95703125, 0.96484375, 0.9765625, 0..."
1,omnimath-551,Qwen2.5-7B-Instruct,"In a game of Fish, R2 and R3 are each holding ...","[To solve this problem, we need to determine t...",True,-1,omnimath,15,"[620, 423, 213, 213, 210, 209, 210, 210, 212, ...","[0.98046875, 0.73046875, 0.953125, 0.99609375,...","[0.2005576797447963, 0.2379298916111697, 0.359...","Two players, R2 and R3, are participating in a...",[In order to find the smallest possible sum of...,True,"[0.9765625, 0.6640625, 0.69140625, 0.87109375,..."
2,olympiadbench-78,Qwen2.5-Math-72B-Instruct,Define $f(x)=\sin ^{6} x+\cos ^{6} x+k\left(\s...,[To solve the equation \( f(x) = 0 \) where \(...,False,3,olympiadbench,8,"[340, 136, 126, 165, 71, 322, 343, 66]","[0.9921875, 0.99609375, 1.0, 0.003997802734375...","[0.5204963202620584, 0.5087271409348658, 0.500...",Define a function \( f(x) \) as \( \sin^6 x + ...,[To solve the equation \( f(x) = 0 \) where \(...,True,"[0.99609375, 1.0, 1.0, 1.0, 0.00433349609375, ..."
3,olympiadbench-832,Llama-3.1-8B-Instruct,The positive integers 34 and 80 have exactly t...,[To find the number of positive integers \( n ...,True,0,olympiadbench,4,"[544, 319, 490, 224]","[0.267578125, 0.15234375, 0.46484375, 0.05078125]","[0.09009299396195182, 0.06853749303199023, 0.1...","Given the positive integers 34 and 80, it is n...","[To address the problem, we start by analyzing...",True,"[0.049560546875, 0.1513671875, 0.17578125, 0.1..."
4,math-184,Llama-3.1-8B-Instruct,Four semi-circles are shown with $AB:BC:CD = 1...,[To find the ratio of the shaded area to the u...,False,1,math,7,"[152, 250, 417, 253, 170, 161, 124]","[1.0, 0.16015625, 0.8046875, 0.062255859375, 0...","[0.2030746281894702, 0.12168575463704703, 0.15...",Four segments of semi-circles are depicted wit...,"[To address this problem, let's begin by defin...",True,"[0.94921875, 0.9765625, 0.9765625, 0.9921875, ..."


In [17]:
index = 4

question, steps = df_sample.iloc[index]["problem"], df_sample.iloc[index]["steps"]
augmented_question, augmented_steps = df_sample.iloc[index]["aug_question"], df_sample.iloc[index]["aug_steps"]
equivalence = df_sample.iloc[index]["equivalence"]
print("\nOriginal Question:")
print("-" * 80)
print(question)

print("\nOriginal Steps:")
print("-" * 80)
print(steps)

print("\nOriginal Rewards:")
print("-" * 80)
print(df_sample.iloc[index]["Qwen/Qwen2.5-Math-PRM-7B--rewards"])

print("\nAugmented Question:")
print("-" * 80)
print(augmented_question)

print("\nAugmented Steps:")
print("-" * 80)
print(augmented_steps)

print("\nAugmented Rewards:")
print("-" * 80)
print(df_sample.iloc[index]["Skywork/Skywork-o1-Open-PRM-Qwen-2.5-7B--rewards"])

print("\nEquivalence Check:")
print("-" * 80)
print(f"Original and augmented versions are equivalent: {equivalence}")




Original Question:
--------------------------------------------------------------------------------
Four semi-circles are shown with $AB:BC:CD = 1:2:3$. What is the ratio of the shaded area to the unshaded area in the semi circle with diameter $AD$? Express your answer as a common fraction. [asy]
import olympiad; import geometry; size(150); defaultpen(linewidth(0.8));
filldraw(arc((6,0),6,0,180)--cycle);
filldraw(arc((3,0),3,0,180)--cycle,fillpen=white); filldraw(arc((8,0),2,0,180)--cycle,fillpen=white); filldraw(arc((11,0),1,0,180)--cycle,fillpen=white);
label("$A$",(12,0),S); label("$B$",(10,0),S); label("$C$",(6,0),S); label("$D$",(0,0),S);
[/asy]

Original Steps:
--------------------------------------------------------------------------------
['To find the ratio of the shaded area to the unshaded area in the semi-circle with diameter AD, we need to first calculate the areas of each semi-circle.'
 'First, determine the radius of each semi-circle. The radius of the semi-circle with 